<div align="center">

# 🚕 Yandex Go Analytics

## 2 · Batch Ingestion

*Read Parquet and Avro files straight from the Volume and persist them in Bronze — as they are.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![ingestion](https://img.shields.io/badge/ingestion-Auto%20Loader-FF3621)
![formats](https://img.shields.io/badge/formats-Parquet%20%2B%20Avro-00ADD4)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [1 · Setup](./1.%20Setup.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) |

**Progress** &nbsp; ▰▰▱▱▱▱▱▱  **2 / 8**

---

## 🎯 Task requirements

- [x] Read the **`.parquet`** and **`.avro`** files directly from the Volume
- [x] Write them **"as-is"** into the `bronze` schema as **Delta** tables: `bronze.taxi` and `bronze.users`

## 🧩 How it is solved

Both tables use the same service, `BaseIngestionService`:

| Step | What happens |
|:--|:--|
| 1 · **Extract** | Auto Loader (`cloudFiles`) reads new files from the Volume; the schema is inferred and stored next to the checkpoint |
| 2 · **Audit columns** | `_ingested_at` (timestamp) and `_source_file` (path) are added — the business columns stay untouched |
| 3 · **Load** | Append to a Delta table with `trigger(availableNow=True)`: process everything available, then stop |

| Entity | Source | Target |
|:--|:--|:--|
| 🚕 Trips | `…/landing/parquet` | `bronze.taxi` |
| 👤 Users | `…/landing/avro` | `bronze.users` |

> 💡 **Incremental by design** — the checkpoint remembers which files were processed, so a re-run ingests only
> *new* files and never duplicates data. Auto Loader also adds a `_rescued_data` column that captures values
> that do not match the inferred schema.

## 1️⃣ Connect to Spark

In [1]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-batch-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 15:20:54.543 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-batch-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_staging | env = staging


## 2️⃣ Peek at the raw sources

Spark can read the files directly from the Volume path — no copy, no staging.

### 🚕 Parquet — trips

In [2]:
SRC_PARQUET = f"/Volumes/{CATALOG}/raw_files/landing/parquet"
SRC_AVRO = f"/Volumes/{CATALOG}/raw_files/landing/avro"

source_trips = spark.read.parquet(SRC_PARQUET)
print(f"Parquet source: {source_trips.count():,} rows | {len(source_trips.columns)} columns")
display(source_trips.toPandas())

Parquet source: 2,000 rows | 22 columns


,id,user_id,driver_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,60cf8db5-261e-409e-a847-44c07c2335e2,ad238d36-dc32-4c97-b9c1-e262f76c8ede,8dca0621-ccfd-437f-9893-63c95a3b2c8a,1,2026-10-07 06:17:29,2026-10-07 06:26:29,4,17.21,3,Y,...,2,57.08,0.0,0.5,8.56,0.00,1.0,69.64,2.5,0.00
1,6aa6576f-2237-43ae-b200-3b65ef4dc8f1,09cb3942-43f5-4a85-bbc9-f87af668a617,6576923c-81e1-4733-91dd-85fdb4164b76,1,2026-10-08 21:25:54,2026-10-08 21:54:54,2,13.92,5,Y,...,1,50.53,0.0,0.5,7.58,12.85,1.0,74.96,2.5,0.00
2,2fa9516d-3ce9-46ae-a9a4-3dec079125cd,98c7472a-864e-4a13-829c-fc0cfa02eaec,8adfffdc-57e0-4e14-ae8b-4cca8736c105,2,2026-10-03 10:00:40,2026-10-03 10:08:40,2,24.74,2,N,...,1,93.36,0.0,0.5,14.00,0.00,1.0,111.36,2.5,0.00
3,47f6c496-20c8-4dd7-b43d-f08683d509ee,4fa03f26-f6f7-40cc-a9ec-8e49d1bdb8c0,3d6350f0-40a5-4f52-af38-a9c7a6c9253a,2,2026-10-05 01:25:17,2026-10-05 01:35:17,1,7.65,1,Y,...,4,28.27,1.0,0.5,4.24,0.00,1.0,37.51,2.5,0.00
4,f18092e3-b9b9-4d0c-8953-aa4093c34281,e4855aa1-016b-4287-b008-05cca7f36ae9,8dca0621-ccfd-437f-9893-63c95a3b2c8a,1,2026-10-08 10:02:39,2026-10-08 10:05:39,3,28.26,3,N,...,3,76.11,0.5,0.5,11.42,0.00,1.0,89.53,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,08c1f54f-3317-4d91-8e10-abf1d8f6484a,2303f6c6-d69d-42f1-ae4c-84ffa8c01f05,f7a31afb-dc59-44a0-99f8-8abeb550f9cf,1,2026-10-01 06:04:53,2026-10-01 06:38:53,4,26.97,1,N,...,1,110.13,0.5,0.5,16.52,0.00,1.0,131.15,2.5,0.00
1996,9aefe046-898d-4cba-b7ff-91cf58278f77,3e2b6091-a092-452a-94a0-57a7b0cc1b3b,a4e33aef-4c48-49ab-b83c-4999147ab562,1,2026-10-07 05:17:51,2026-10-07 05:47:51,6,19.45,4,Y,...,4,60.64,1.0,0.5,0.00,8.90,1.0,74.54,2.5,0.00
1997,3dde3575-49f0-4335-adf0-9b2ff2275e7c,10f1bc81-448a-4a9e-a6b2-bc5b50c187fc,ba7858bb-61d4-41c5-a16c-5c3c2cab43d8,2,2026-10-06 03:24:10,2026-10-06 04:03:10,1,21.37,5,N,...,1,75.95,2.5,0.5,0.00,0.00,1.0,82.45,2.5,0.00
1998,e4a15718-323e-424d-aa15-49d82d61c862,bdd640fb-0667-4ad1-9c80-317fa3b1799d,299c3bcf-90ef-4625-99fb-cdb5c6042444,2,2026-10-08 03:10:58,2026-10-08 03:38:58,5,8.66,2,N,...,3,26.77,2.5,0.5,4.02,0.00,1.0,38.54,2.5,1.25


### 👤 Avro — users

In [3]:
source_users = spark.read.format("avro").load(SRC_AVRO)
print(f"Avro source: {source_users.count():,} rows | {len(source_users.columns)} columns")
display(source_users.toPandas())

Avro source: 500 rows | 5 columns


,id,full_name,email,registration_date,is_plus_subscriber
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Rachel Castillo,melissa79@example.net,2021-08-22,True
1,23b8c1e9-3924-46de-beb1-3b9046685257,Sarah Rodriguez,roger12@example.org,2025-12-15,True
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,Benjamin West,robbinsandrea@example.org,2021-07-19,False
3,972a8469-1641-4f82-8b9d-2434e465e150,Brenda Hodges,dcox@example.org,2020-03-15,True
4,17fc695a-07a0-4a6e-8822-e8f36c031199,Alfred Campbell,joseph90@example.org,2023-08-10,False
...,...,...,...,...,...
495,d56ce8ea-1959-4b5a-a7a8-f6366a35df59,April Terry,monicareyes@example.net,2023-02-08,False
496,f065c817-687a-4850-9d9b-62317d45d8ef,Michael Marshall,xkelley@example.com,2026-01-14,True
497,b0b63bcf-0860-4833-879d-0cdaf396ea37,Teresa Mcmahon,edwardyoder@example.net,2025-12-05,True
498,71d79665-7181-4dcf-b79e-fc6e5edb0d3c,Jimmy Clark,michelle83@example.net,2020-01-20,True


## 3️⃣ Ingest into Bronze

Two services, one pattern. Each call blocks until the available files are fully processed.

```text
 landing/avro      ──▶ UserBatchIngestionService      ──▶ bronze.users
 landing/parquet   ──▶ TaxiTripBatchIngestionService  ──▶ bronze.taxi
```

In [4]:
from src.core.etl.bronze import (
    TaxiTripBatchIngestionService,
    UserBatchIngestionService,
)

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

print("✅ Batch ingestion finished")

2026-10-09 15:21:02.649 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_staging.bronze.users
2026-10-09 15:21:02.651 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [avro] from: /Volumes/yandex_go_staging/raw_files/landing/avro
2026-10-09 15:21:02.813 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_staging.bronze.users
2026-10-09 15:21:19.605 | INFO     | src.core.etl.bronze.base:load:111 - Successfully loaded in yandex_go_staging.bronze.users
2026-10-09 15:21:19.608 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_staging.bronze.taxi
2026-10-09 15:21:19.609 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [parquet] from: /Volumes/yandex_go_staging/raw_files/landing/parquet
2026-10-09 15:21:19.610 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_staging.bronze.taxi
2026-10-09 15:21:31.201 | INFO     | src.core.e

✅ Batch ingestion finished


---

## ✅ Checkpoint

- [x] Parquet and Avro files read directly from the Volume
- [x] `bronze.taxi` and `bronze.users` written as Delta tables
- [x] Row counts match the sources; re-running adds nothing

> **Next up ▶ [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb)** — stream drivers into Bronze, evolve the schema and travel through time.

<div align="center"><sub>Yandex Go Analytics · notebook 2 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>